# Agent 工作流与 RAG 检索流程演示

本 Notebook 演示两大核心主题：
1. **RAG 检索流程**：文档加载 → 文本切分 → Embedding 向量化 → FAISS 相似度检索 → LLM 生成答案。
2. **Agent 工作流（ReAct 风格）**：大语言模型根据问题选择工具（检索、计算器等），通过“思考-行动-观察”多步推理得到最终答案。

> 运行前请确保已安装依赖：`uv pip install -r requirements.txt` 或手动安装 `langchain`, `langchain-community`, `langchain-openai`, `faiss-cpu`, `openai`。
# 本示例使用智谱 AI API 完成 Embedding 和对话，无需本地 GPU 或 torch。

## 1. 安装与导入依赖

## 0. 读取配置文件

本示例所有 Embedding 和 LLM 配置均从当前目录下的 `config.json` 文件读取。请创建该文件，内容格式如下：

```json
{
  "zhipu_api_key": "你的智谱 API Key",
  "chat_model": "glm-4-flash",
  "embedding_model": "embedding-3"
}
```

In [39]:
import json
import os

CONFIG_FILE = "config.json"

# 默认配置
config = {
    "zhipu_api_key": "your-zhipu-api-key-here",
    "chat_model": "glm-4-flash",
    "embedding_model": "embedding-3"
}

# 从 config.json 读取配置（如果存在则覆盖默认值）
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        config.update(json.load(f))

ZHIPU_API_KEY = config["zhipu_api_key"]
CHAT_MODEL = config["chat_model"]
EMBEDDING_MODEL = config["embedding_model"]

print(f"智谱 API Key 是否已设置: {bool(ZHIPU_API_KEY and not ZHIPU_API_KEY.startswith('your-'))}")
print(f"聊天模型: {CHAT_MODEL}")
print(f"Embedding 模型: {EMBEDDING_MODEL}")

智谱 API Key 是否已设置: True
聊天模型: glm-4.6v-flashx
Embedding 模型: embedding-3


In [28]:
# 可选：在当前 kernel 环境中安装依赖（如果尚未安装）
# 如果已经在 .venv 中安装过，可跳过此单元格
# !uv pip install langchain langchain-community langchain-text-splitters faiss-cpu zhipuai pyjwt==2.8.0 cachetools

In [40]:
import json
import os
import re
from typing import List, Optional

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.embeddings import Embeddings
from zhipuai import ZhipuAI

print("依赖导入完成")

依赖导入完成


In [41]:
# 定义智谱 AI 的 Embedding 适配器
class ZhipuAIEmbeddings(Embeddings):
    def __init__(self, api_key: str, model: str = "embedding-3"):
        self.client = ZhipuAI(api_key=api_key)
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        resp = self.client.embeddings.create(model=self.model, input=texts)
        return [item.embedding for item in resp.data]

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]


# 定义智谱 AI 的聊天模型适配器
class ChatZhipuAI(BaseChatModel):
    api_key: str
    model: str = "glm-4-flash"
    temperature: float = 0.7
    max_tokens: int = 1024

    def __init__(self, api_key: str, model: str = "glm-4-flash", temperature: float = 0.7, max_tokens: int = 1024, **kwargs):
        super().__init__(api_key=api_key, model=model, temperature=temperature, max_tokens=max_tokens, **kwargs)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        client = ZhipuAI(api_key=self.api_key)
        zhipu_messages = []
        for m in messages:
            if isinstance(m, SystemMessage):
                zhipu_messages.append({"role": "system", "content": m.content})
            elif isinstance(m, HumanMessage):
                zhipu_messages.append({"role": "user", "content": m.content})
            elif isinstance(m, AIMessage):
                zhipu_messages.append({"role": "assistant", "content": m.content or ""})
            elif isinstance(m, ToolMessage):
                zhipu_messages.append({"role": "tool", "content": m.content, "tool_call_id": m.tool_call_id})

        response = client.chat.completions.create(
            model=self.model,
            messages=zhipu_messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        content = response.choices[0].message.content
        message = AIMessage(content=content)
        generation = ChatGeneration(message=message)
        return ChatResult(generations=[generation])

    def _llm_type(self) -> str:
        return "zhipuai"

print("智谱适配器定义完成")

智谱适配器定义完成


## 2. 加载与切分文档

RAG 的第一步是准备知识库。这里我们直接用字符串模拟一篇企业知识文档；你也可以替换为 `TextLoader`、`WebBaseLoader` 或 `UnstructuredFileLoader`。

In [42]:
# 模拟知识库文档
raw_text = """
公司假期政策：
1. 年假：入职满 1 年可享受 5 天带薪年假，每多 1 年增加 1 天，上限 15 天。
2. 病假：每年最多 10 天，需提供医院证明。
3. 产假：女性员工可享受 158 天产假，男性员工可享受 15 天陪产假。
4. 调休：加班可申请调休，调休有效期为 6 个月。

报销流程：
1. 员工在 OA 系统提交报销单，并上传发票扫描件。
2. 直属领导审批后，财务部门在 5 个工作日内完成打款。
3. 单笔超过 5000 元的发票需要副总裁签字。

技术支持：
1. 工作电脑故障请拨打 400-888-1234 或发送邮件到 it-support@example.com。
2. VPN 连接问题请参考《远程办公手册》第三章。
3. 内部系统账号锁定后，可在登录页点击“忘记密码”自助重置。
"""

docs = [Document(page_content=raw_text, metadata={"source": "company_handbook"})]

# 文本切分
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", "。", " ", ""]
)
chunks = text_splitter.split_documents(docs)

print(f"切分后文档块数量: {len(chunks)}")
for i, c in enumerate(chunks[:5]):
    print(f"\n--- 块 {i+1} ---\n{c.page_content[:120]}...")

切分后文档块数量: 3

--- 块 1 ---
公司假期政策：
1. 年假：入职满 1 年可享受 5 天带薪年假，每多 1 年增加 1 天，上限 15 天。
2. 病假：每年最多 10 天，需提供医院证明。
3. 产假：女性员工可享受 158 天产假，男性员工可享受 15 天陪产假。
4...

--- 块 2 ---
报销流程：
1. 员工在 OA 系统提交报销单，并上传发票扫描件。
2. 直属领导审批后，财务部门在 5 个工作日内完成打款。
3. 单笔超过 5000 元的发票需要副总裁签字。...

--- 块 3 ---
技术支持：
1. 工作电脑故障请拨打 400-888-1234 或发送邮件到 it-support@example.com。
2. VPN 连接问题请参考《远程办公手册》第三章。
3. 内部系统账号锁定后，可在登录页点击“忘记密码”自助重置...


## 3. 配置 Embedding 模型与向量存储

将切分后的文本块转换为向量，并构建 FAISS 向量数据库。这里通过 OpenAI API 获取 Embedding，无需本地模型。

In [43]:
# 初始化智谱 Embedding 模型
embedding_model = ZhipuAIEmbeddings(
    api_key=ZHIPU_API_KEY,
    model=EMBEDDING_MODEL
)

# 构建 FAISS 向量数据库
vectorstore = FAISS.from_documents(chunks, embedding_model)

print("FAISS 向量数据库构建完成")
print(f"向量维度: {vectorstore.index.d}")
print(f"索引中文本块数量: {vectorstore.index.ntotal}")

FAISS 向量数据库构建完成
向量维度: 2048
索引中文本块数量: 3


## 4. 创建 RAG 检索器

将向量数据库包装为检索器，配置返回最相关的 top-k 个文本块。

In [44]:
# 创建检索器，返回 top-3 最相关文档
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "年假有多少天？"
retrieved_docs = retriever.invoke(query)

print(f"查询: {query}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- 结果 {i+1} ---")
    print(doc.page_content)
    print(doc.metadata, "\n")

查询: 年假有多少天？

--- 结果 1 ---
公司假期政策：
1. 年假：入职满 1 年可享受 5 天带薪年假，每多 1 年增加 1 天，上限 15 天。
2. 病假：每年最多 10 天，需提供医院证明。
3. 产假：女性员工可享受 158 天产假，男性员工可享受 15 天陪产假。
4. 调休：加班可申请调休，调休有效期为 6 个月。
{'source': 'company_handbook'} 

--- 结果 2 ---
报销流程：
1. 员工在 OA 系统提交报销单，并上传发票扫描件。
2. 直属领导审批后，财务部门在 5 个工作日内完成打款。
3. 单笔超过 5000 元的发票需要副总裁签字。
{'source': 'company_handbook'} 

--- 结果 3 ---
技术支持：
1. 工作电脑故障请拨打 400-888-1234 或发送邮件到 it-support@example.com。
2. VPN 连接问题请参考《远程办公手册》第三章。
3. 内部系统账号锁定后，可在登录页点击“忘记密码”自助重置。
{'source': 'company_handbook'} 



## 5. 定义 Agent 可调用的工具

把检索能力、计算能力等封装成工具函数。ReAct Agent 会根据问题自动判断调用哪个工具。

In [45]:
@tool
def company_knowledge_retriever(question: str) -> str:
    """
    检索公司内部知识库，回答关于假期、报销、IT 支持等问题。
    输入应为一句完整的问题，例如："年假有多少天？"
    """
    docs = retriever.invoke(question)
    return "\n\n".join([d.page_content for d in docs])


@tool
def calculator(expression: str) -> str:
    """
    计算数学表达式，例如："5 + 3 * 2"、"158 / 7"。
    """
    try:
        result = eval(expression)
        return f"计算结果: {result}"
    except Exception as e:
        return f"计算失败: {e}"


@tool
def search_web(query: str) -> str:
    """
    模拟网页搜索工具。实际项目中可替换为 Tavily、DuckDuckGo 或 Bing Search API。
    """
    return f"[模拟网页搜索结果] 关于 '{query}' 的最新公开信息：……（这里是搜索结果摘要）"


tools = [company_knowledge_retriever, calculator, search_web]
print(f"已注册 {len(tools)} 个工具：{[t.name for t in tools]}")

已注册 3 个工具：['company_knowledge_retriever', 'calculator', 'search_web']


## 6. 配置大语言模型（LLM）

这里使用智谱 AI 的聊天模型。你可以根据需求切换为 `glm-4-flash`（免费/低成本）、`glm-4`、`glm-4-plus` 等。

In [46]:
# 初始化智谱大语言模型
llm = ChatZhipuAI(
    api_key=ZHIPU_API_KEY,
    model=CHAT_MODEL,  # 从 config.json 读取，默认 glm-4-flash
    temperature=0.0,
    max_tokens=1024
)
print("LLM 初始化完成")
print(f"使用模型: {llm.model}")

LLM 初始化完成
使用模型: glm-4.6v-flashx


## 7. 构建 ReAct Agent 工作流

使用智谱 LLM 手动实现 ReAct 循环：

```
Question -> Thought -> Action -> Observation -> ... -> Final Answer
```

循环逻辑：
1. 把 system prompt、用户问题、历史观察拼接成上下文。
2. 调用智谱 LLM，要求它以 `Action: 工具名\nAction Input: 输入` 或 `Final Answer: 答案` 格式回复。
3. 若输出包含 Action，则执行对应工具，把结果作为 Observation 追加到上下文。
4. 重复 2-3 直到输出 Final Answer。

In [47]:
# 手动实现 ReAct Agent（适配智谱 API）
import re

class ReActAgent:
    def __init__(self, llm, tools, system_prompt: str, max_iterations: int = 5):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations

    def _call_llm(self, prompt: str) -> str:
        response = self.llm._generate([HumanMessage(content=prompt)])
        return response.generations[0].message.content

    def _parse_action(self, text: str):
        """从 LLM 输出中解析 Action / Action Input / Final Answer"""
        if "Final Answer:" in text:
            return "final", text.split("Final Answer:", 1)[1].strip()
        action_match = re.search(r"Action:\s*(\w+)", text)
        input_match = re.search(r"Action Input:\s*(.+?)(?=\n(?:Observation|Thought|Action|Final Answer)|$)", text, re.DOTALL)
        if action_match:
            action = action_match.group(1)
            action_input = input_match.group(1).strip() if input_match else ""
            return "action", (action, action_input)
        return "unknown", text

    def run(self, question: str) -> str:
        scratchpad = f"Question: {question}\nThought: 我需要先分析问题，决定使用哪个工具。"
        for i in range(self.max_iterations):
            prompt = f"""{self.system_prompt}

当前问题与历史记录：
{scratchpad}

请用以下格式之一回复：
Thought: 你的思考
Action: 工具名称（可选：{', '.join(self.tools.keys())}）
Action Input: 工具的输入

或当你已有最终答案时：
Thought: 我已知道最终答案
Final Answer: 最终答案
"""
            output = self._call_llm(prompt)
            print(f"\n--- Step {i+1} ---")
            print(output)

            kind, payload = self._parse_action(output)
            if kind == "final":
                return payload

            if kind == "action":
                action_name, action_input = payload
                if action_name not in self.tools:
                    observation = f"工具 '{action_name}' 不存在，可用工具为：{list(self.tools.keys())}"
                else:
                    observation = self.tools[action_name].invoke(action_input)
                scratchpad += f"\n{output}\nObservation: {observation}\nThought: 让我继续分析。"
            else:
                scratchpad += f"\n{output}\nObservation: 无法解析你的回复，请使用 Action/Action Input 或 Final Answer 格式。"

        return "达到最大迭代次数，未能获得最终答案。"


agent_system_prompt = """你是一个智能助手，可以使用以下工具回答用户问题：

1. company_knowledge_retriever：检索公司内部知识库，回答假期、报销、IT 支持等问题。
2. calculator：计算数学表达式。
3. search_web：模拟网页搜索（当前为演示用途）。

请遵循以下原则：
- 如果问题涉及公司内部政策，优先调用 company_knowledge_retriever。
- 如果需要数值计算，调用 calculator。
- 用中文回答用户。"""

agent = ReActAgent(llm=llm, tools=tools, system_prompt=agent_system_prompt)
print("ReAct Agent 构建完成")

ReAct Agent 构建完成


## 8. 运行 RAG 检索流程演示

单独展示 RAG 链路：用户提问 → 向量检索 → 结果传入 LLM → 生成答案。

In [48]:
rag_query = "报销超过 5000 元需要什么审批？"

# 1. 检索相关文档片段
retrieved = retriever.invoke(rag_query)
context = "\n\n".join([d.page_content for d in retrieved])

# 2. 构造提示词
rag_prompt = f"""请根据以下参考资料回答问题。如果资料中找不到答案，请说明无法回答。

参考资料：
{context}

问题：{rag_query}
"""

# 3. 调用 LLM 生成答案
response = llm.invoke(rag_prompt)

print("=" * 40)
print("RAG 检索到的上下文：")
print(context)
print("=" * 40)
print("\n最终答案：")
print(response.content)

RAG 检索到的上下文：
报销流程：
1. 员工在 OA 系统提交报销单，并上传发票扫描件。
2. 直属领导审批后，财务部门在 5 个工作日内完成打款。
3. 单笔超过 5000 元的发票需要副总裁签字。

公司假期政策：
1. 年假：入职满 1 年可享受 5 天带薪年假，每多 1 年增加 1 天，上限 15 天。
2. 病假：每年最多 10 天，需提供医院证明。
3. 产假：女性员工可享受 158 天产假，男性员工可享受 15 天陪产假。
4. 调休：加班可申请调休，调休有效期为 6 个月。

技术支持：
1. 工作电脑故障请拨打 400-888-1234 或发送邮件到 it-support@example.com。
2. VPN 连接问题请参考《远程办公手册》第三章。
3. 内部系统账号锁定后，可在登录页点击“忘记密码”自助重置。

最终答案：

根据参考资料，报销超过5000元的发票需要副总裁签字。


## 9. 运行 Agent 多步推理演示

提出一个需要结合**检索**和**计算**的问题，观察 Agent 如何选择工具并完成推理。

In [49]:
agent_query = "我入职第 3 年，请问我这一年能休多少天年假？如果把这些年假平均分配到 12 个月，每个月大约能休几天？"

final_answer = agent.run(agent_query)

print("\n" + "=" * 40)
print("Agent 最终答案：")
print(final_answer)


--- Step 1 ---

Thought: 我需要先查询公司内部关于年假的政策，因为问题涉及公司内部政策，所以优先调用company_knowledge_retriever来获取相关信息。
Action: company_knowledge_retriever
Action Input: 入职第3年年假天数及分配规则

--- Step 2 ---

Thought: 我需要先确定入职第3年的年假天数，根据公司政策“入职满1年可享受5天带薪年假，每多1年增加1天”，所以入职第3年可休5 + (3 - 1) = 7天年假。接下来需要计算这7天年假平均分配到12个月每月的天数，调用calculator计算7除以12的结果。
Action: calculator
Action Input: 7/12

--- Step 3 ---

Thought: 我已知道最终答案
Final Answer: 入职第3年可休7天年假，平均分配到12个月每月大约能休0.58天（约等于7/12的结果）。

Agent 最终答案：
入职第3年可休7天年假，平均分配到12个月每月大约能休0.58天（约等于7/12的结果）。


---

## 总结

| 流程 | 关键步骤 | 说明 |
|------|----------|------|
| **RAG** | 文档 → 切分 → Embedding → 向量检索 → LLM 生成 | 让 LLM 基于私有知识库回答问题，减少幻觉 |
| **Agent** | Question → Thought → Action → Observation → Final Answer | LLM 自主决策使用哪些工具，完成多步任务 |

本示例中的 `search_web` 是模拟工具，真实场景可替换为搜索引擎 API；`calculator` 可替换为 `numexpr` 或代码解释器以提升安全性。